In [1]:
import os
import requests
import datetime
from zoneinfo import ZoneInfo
from dotenv import load_dotenv
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

# Load secrets from .env
load_dotenv()

API_KEY = os.getenv("ALPACA_API_KEY")
SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")
DISCORD_WEBHOOK_URL = os.getenv("DISCORD_WEBHOOK_URL")

print("Keys loaded:" if API_KEY and SECRET_KEY and DISCORD_WEBHOOK_URL else "⚠️ Missin' some .env vars, boss!")
print(f"Webhook URL ends with: ...{DISCORD_WEBHOOK_URL[-20:] if DISCORD_WEBHOOK_URL else 'None'}")

# Client for Alpaca data
client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

ModuleNotFoundError: No module named 'alpaca'

In [1]:
def calculate_pivots(high, low, close):
    pp = (high + low + close) / 3
    r1 = 2 * pp - low
    s1 = 2 * pp - high
    r2 = pp + (high - low)
    s2 = pp - (high - low)
    r3 = pp + 2 * (high - low)
    s3 = pp - 2 * (high - low)
    return {k: round(v, 2) for k, v in {
        "PP": pp, "R1": r1, "S1": s1, "R2": r2, "S2": s2, "R3": r3, "S3": s3
    }.items()}

def get_yesterdays_spx():
    est = ZoneInfo("America/New_York")
    end_date = datetime.datetime.now(est).date() - datetime.timedelta(days=1)
    start_date = end_date - datetime.timedelta(days=20)
    
    symbols = ["SPX", "^GSPC"]
    for symbol in symbols:
        try:
            request_params = StockBarsRequest(
                symbol_or_symbols=symbol,
                timeframe=TimeFrame.Day,
                start=start_date,
                end=end_date,
                adjustment="all"
            )
            bars = client.get_stock_bars(request_params).df
            if not bars.empty:
                bars = bars[bars.index.date <= end_date]
                latest = bars.iloc[-1]
                return {
                    "high": latest["high"],
                    "low": latest["low"],
                    "close": latest["close"],
                    "symbol_used": symbol
                }
        except Exception as e:
            print(f"{symbol} failed: {e}")
    raise ValueError("Couldn't pull SPX data – check keys/feed")

# Run the test
data = get_yesterdays_spx()
pivots = calculate_pivots(data["high"], data["low"], data["close"])

print(f"Data pulled using symbol: {data['symbol_used']}")
print(f"High: {data['high']}, Low: {data['low']}, Close: {data['close']}")
print("Pivots:")
for k, v in pivots.items():
    print(f"  {k}: {v}")

NameError: name 'ZoneInfo' is not defined

In [2]:
def send_discord_message(message, dry_run=True):
    if dry_run:
        print("=== DISCORD MESSAGE (dry run) ===\n")
        print(message)
        print("\n=== END MESSAGE ===\n")
    else:
        response = requests.post(DISCORD_WEBHOOK_URL, json={"content": message})
        response.raise_for_status()
        print("Message sent to Discord!")

def send_greeting(dry_run=True):
    data = get_yesterdays_spx()
    pivots = calculate_pivots(data["high"], data["low"], data["close"])
    
    message = (
        "Yo, mornin' you beautiful degenerates! Snoogans here, fresh off a coffee that's blacker than Silent Bob's soul.\n"
        f"Today's SPX intraday pivots are locked and loaded: Pivot Point at {pivots['PP']}, "
        f"R1 {pivots['R1']}, S1 {pivots['S1']}, R2 {pivots['R2']}, S2 {pivots['S2']}, "
        f"R3 {pivots['R3']}, S3 {pivots['S3']}.\n"
        "We're talkin' 0DTE/1DTE vertical credit spreads only—mostly 16-delta puts to kick it off, calls if IV gets all spicy. "
        "Hard 50% profit target, 2x stop loss, 4pm cutoff, max 3% daily nanny shutdown. "
        "Sizin' starts at 1-3 contracts, growin' like a responsible adult.\n"
        "Alpaca feed locked, secrets safe—let's sell some tiny spreads and secure that lunch money, snoochie boochies!"
    )
    send_discord_message(message, dry_run=dry_run)

# Test it – set dry_run=False if you're feelin' brave and wanna send for real
send_greeting(dry_run=True)

NameError: name 'ZoneInfo' is not defined

In [3]:
def send_entry(is_put_spread=True, short_strike=6820, long_strike=6810, credit=2.50, underlying="SPX", dry_run=True):
    spread_type = "put" if is_put_spread else "call"
    strikes = f"{short_strike}/{long_strike}"
    message = (
        f"Just sold a tiny {spread_type} credit spread on {underlying}, "
        f"strikes {strikes}, credit {credit} bucks.\n"
        "Defined risk only, no naked nonsense—PPO brain's feelin' zen on that Alpaca real-time juice.\n"
        "Account's growin' slow and steady, paper to real money once it proves it's not an idiot.\n"
        "Snoogans banks another one, lunch money secured, snoochie boochies!"
    )
    send_discord_message(message, dry_run=dry_run)

# Try a put spread
send_entry(is_put_spread=True, short_strike=6020, long_strike=6000, credit=3.15, dry_run=True)

# Try a call spread
send_entry(is_put_spread=False, short_strike=6050, long_strike=6070, credit=2.80, dry_run=True)

=== DISCORD MESSAGE (dry run) ===

Just sold a tiny put credit spread on SPX, strikes 6020/6000, credit 3.15 bucks.
Defined risk only, no naked nonsense—PPO brain's feelin' zen on that Alpaca real-time juice.
Account's growin' slow and steady, paper to real money once it proves it's not an idiot.
Snoogans banks another one, lunch money secured, snoochie boochies!

=== END MESSAGE ===

=== DISCORD MESSAGE (dry run) ===

Just sold a tiny call credit spread on SPX, strikes 6050/6070, credit 2.8 bucks.
Defined risk only, no naked nonsense—PPO brain's feelin' zen on that Alpaca real-time juice.
Account's growin' slow and steady, paper to real money once it proves it's not an idiot.
Snoogans banks another one, lunch money secured, snoochie boochies!

=== END MESSAGE ===



In [4]:
def send_exit(is_put_spread=True, short_strike=6820, long_strike=6810, original_credit=2.50, pnl=+1.25, underlying="SPX", dry_run=True):
    spread_type = "put" if is_put_spread else "call"
    strikes = f"{short_strike}/{long_strike}"
    result = "win" if pnl > 0 else "loss"
    message = (
        f"Closed out that {spread_type} credit spread, strikes {strikes}, "
        f"original credit {original_credit}, exit for {pnl:+.2f} (that's a {result}, degenerates!).\n"
        "Hit the 50% profit target / 2x stop / time cutoff without panickin'—"
        "wholesome RL agent never got yelled at, just chill compoundin' on Alpaca.\n"
        "Daily vibes: +[percentage] so far, drawdown barely a blip.\n"
        "Snoogans out, snoochie boochies—time to watch Clerks and plot the next tiny theta grab!"
    )
    send_discord_message(message, dry_run=dry_run)

# Green day
send_exit(pnl=+1.25, dry_run=True)

# Small loss (still defined risk, mom proud)
send_exit(pnl=-4.80, original_credit=2.50, dry_run=True)

=== DISCORD MESSAGE (dry run) ===

Closed out that put credit spread, strikes 6820/6810, original credit 2.5, exit for +1.25 (that's a win, degenerates!).
Hit the 50% profit target / 2x stop / time cutoff without panickin'—wholesome RL agent never got yelled at, just chill compoundin' on Alpaca.
Daily vibes: +[percentage] so far, drawdown barely a blip.
Snoogans out, snoochie boochies—time to watch Clerks and plot the next tiny theta grab!

=== END MESSAGE ===

=== DISCORD MESSAGE (dry run) ===

Closed out that put credit spread, strikes 6820/6810, original credit 2.5, exit for -4.80 (that's a loss, degenerates!).
Hit the 50% profit target / 2x stop / time cutoff without panickin'—wholesome RL agent never got yelled at, just chill compoundin' on Alpaca.
Daily vibes: +[percentage] so far, drawdown barely a blip.
Snoogans out, snoochie boochies—time to watch Clerks and plot the next tiny theta grab!

=== END MESSAGE ===

